# Nuclear Data

In this notebook you will learn how to:
- Load a cross-section directory (xsdir) and look up nuclides
- Query pointwise cross sections and plot them vs. energy
- Build materials and compute macroscopic quantities
- Collapse to multigroup constants and visualize them
- Doppler-broaden cross sections and see the effect on resonances
- Compare cross sections across nuclides

**Prerequisites:** ACE-format nuclear data files and an xsdir index.

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from aleathor.nucdata import XsDir, Nuclide, NucMaterial, Multigroup, parse_zaid

## Load cross-section directory

Point `XsDir` to your xsdir file. Relative paths in the xsdir are resolved from the file's directory.

In [ ]:
xsdir = XsDir("/path/to/xsdir")
print(xsdir)

## Load nuclides

In [ ]:
u235 = Nuclide(xsdir, "92235.80c")
u238 = Nuclide(xsdir, "92238.80c")
o16  = Nuclide(xsdir, "8016.80c")
h1   = Nuclide(xsdir, "1001.80c")

for nuc in [u235, u238, o16, h1]:
    print(f"{nuc}  AWR={nuc.awr:.3f}  fissile={nuc.is_fissile}  "
          f"energies={nuc.n_energies}  reactions={nuc.n_reactions}")

## Pointwise cross sections vs. energy

Plot total, absorption, and elastic cross sections over the full energy range.

In [ ]:
energies = np.geomspace(1e-10, 20, 5000)  # MeV

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, nuc, title in zip(axes, [u235, u238], ["U-235", "U-238"]):
    xs_t = [nuc.xs_total(e) for e in energies]
    xs_a = [nuc.xs_absorption(e) for e in energies]
    xs_e = [nuc.xs_elastic(e) for e in energies]

    ax.loglog(energies, xs_t, label="Total", lw=0.5)
    ax.loglog(energies, xs_a, label="Absorption", lw=0.5)
    ax.loglog(energies, xs_e, label="Elastic", lw=0.5)
    ax.set_xlabel("Energy (MeV)")
    ax.set_ylabel("Cross section (barn)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.show()

## Comparing nuclides

Total cross section for several common nuclides on the same plot.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for nuc, label in [(h1, "H-1"), (o16, "O-16"), (u235, "U-235"), (u238, "U-238")]:
    xs = [nuc.xs_total(e) for e in energies]
    ax.loglog(energies, xs, label=label, lw=0.6)

ax.set_xlabel("Energy (MeV)")
ax.set_ylabel("Total cross section (barn)")
ax.set_title("Total cross section comparison")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

## Fission: nu-bar vs. energy

Average number of neutrons emitted per fission for U-235.

In [ ]:
e_fiss = np.geomspace(1e-8, 20, 500)
nu = [u235.nu_bar(e) for e in e_fiss]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(e_fiss, nu, lw=1.2)
ax.set_xlabel("Energy (MeV)")
ax.set_ylabel(r"$\bar{\nu}$")
ax.set_title("U-235 average neutrons per fission")
ax.grid(True, which="both", alpha=0.3)
plt.show()

## Material: macroscopic cross sections

Build a UO2 fuel material (4.5 % enriched, 10.97 g/cm³) and plot macroscopic cross sections.

In [ ]:
# Number densities for UO2 at 10.97 g/cm³, 4.5% enriched
uo2 = NucMaterial()
uo2.add(u235, 0.001105)   # atoms/barn-cm
uo2.add(u238, 0.02234)
uo2.add(o16,  0.04689)

macro_t = [uo2.xs_total(e) for e in energies]
macro_a = [uo2.xs_absorption(e) for e in energies]
mfp     = [uo2.mean_free_path(e) for e in energies]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.loglog(energies, macro_t, label=r"$\Sigma_t$", lw=0.6)
ax1.loglog(energies, macro_a, label=r"$\Sigma_a$", lw=0.6)
ax1.set_xlabel("Energy (MeV)")
ax1.set_ylabel(r"Macroscopic XS (cm$^{-1}$)")
ax1.set_title("UO2 macroscopic cross sections")
ax1.legend()
ax1.grid(True, which="both", alpha=0.3)

ax2.loglog(energies, mfp, lw=0.6, color="C2")
ax2.set_xlabel("Energy (MeV)")
ax2.set_ylabel("Mean free path (cm)")
ax2.set_title("UO2 mean free path")
ax2.grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.show()

## Multigroup collapse

Collapse U-235 continuous-energy data into a few-group structure and visualize the group constants.

In [ ]:
# CASMO-style 4-group boundaries (MeV, descending)
bounds = [20.0, 0.821, 5.53e-3, 6.25e-7, 1e-11]
mg = Multigroup(bounds)
mg.collapse(u235)

data = mg.get_data()
ng = data["n_groups"]

print(f"{mg}\n")
for g in range(ng):
    print(f"  Group {g+1}: [{bounds[g]:.3e}, {bounds[g+1]:.3e}] MeV")
    print(f"    sigma_t={data['sigma_t'][g]:.3f}  sigma_a={data['sigma_a'][g]:.3f}  "
          f"nu_sigma_f={data['nu_sigma_f'][g]:.3f}  chi={data['chi'][g]:.5f}")

In [ ]:
group_labels = [f"G{g+1}" for g in range(ng)]
x = np.arange(ng)
w = 0.2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of group constants
ax1.bar(x - w, data["sigma_t"], width=w, label=r"$\sigma_t$")
ax1.bar(x,     data["sigma_a"], width=w, label=r"$\sigma_a$")
ax1.bar(x + w, data["nu_sigma_f"], width=w, label=r"$\nu\sigma_f$")
ax1.set_xticks(x)
ax1.set_xticklabels(group_labels)
ax1.set_ylabel("Cross section (barn)")
ax1.set_title("U-235 multigroup constants")
ax1.legend()
ax1.set_yscale("log")
ax1.grid(True, axis="y", alpha=0.3)

# Fission spectrum (chi)
ax2.bar(x, data["chi"], color="C3")
ax2.set_xticks(x)
ax2.set_xticklabels(group_labels)
ax2.set_ylabel(r"$\chi_g$")
ax2.set_title("Fission spectrum")
ax2.grid(True, axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

### Scattering matrix

Visualize the group-to-group scattering matrix as a heatmap.

In [ ]:
scat_matrix = np.array([[mg.scatter(gf, gt) for gt in range(ng)] for gf in range(ng)])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(scat_matrix, origin="upper", cmap="viridis")
ax.set_xticks(range(ng))
ax.set_yticks(range(ng))
ax.set_xticklabels(group_labels)
ax.set_yticklabels(group_labels)
ax.set_xlabel("To group")
ax.set_ylabel("From group")
ax.set_title(r"U-235 scattering matrix $\sigma_{s}(g \to g')$")

# Annotate cells with values
for gf in range(ng):
    for gt in range(ng):
        val = scat_matrix[gf, gt]
        color = "white" if val < scat_matrix.max() / 2 else "black"
        ax.text(gt, gf, f"{val:.2f}", ha="center", va="center", color=color, fontsize=9)

fig.colorbar(im, ax=ax, label="barn")
fig.tight_layout()
plt.show()

## Doppler broadening

Load a fresh (non-cached) copy of U-238 and broaden it to 600 K. Compare the resolved resonance region before and after.

In [ ]:
# Fresh copy for broadening (cached=False)
u238_hot = Nuclide(xsdir, "92238.80c", cached=False)
print(f"Original temperature: {u238_hot.temperature:.4e} MeV")

kT_600K = 5.17e-8  # 600 K in MeV
u238_hot.doppler_broaden(kT_600K)
print(f"After broadening:    {u238_hot.temperature:.4e} MeV")

In [ ]:
# Zoom into the resolved resonance region around the 6.67 eV resonance
e_res = np.geomspace(1e-6, 1e-4, 3000)  # MeV (1 eV to 100 eV)

xs_cold = [u238.xs_total(e) for e in e_res]
xs_hot  = [u238_hot.xs_total(e) for e in e_res]

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(e_res * 1e6, xs_cold, label="Cold (original)", lw=0.7)
ax.loglog(e_res * 1e6, xs_hot,  label="Broadened (600 K)", lw=0.7)
ax.set_xlabel("Energy (eV)")
ax.set_ylabel("Total cross section (barn)")
ax.set_title("U-238 Doppler broadening — resolved resonance region")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

## Monte Carlo sampling demo

Sample collision distances and target nuclides in UO2 at thermal energy.

In [ ]:
import random

e_thermal = 0.0253e-6  # 0.0253 eV in MeV
n_samples = 10_000

distances = [uo2.sample_distance(e_thermal, random.random()) for _ in range(n_samples)]
nuclides  = [uo2.sample_nuclide(e_thermal, random.random())[1] for _ in range(n_samples)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distance distribution (should be exponential)
ax1.hist(distances, bins=80, density=True, alpha=0.7, edgecolor="black", lw=0.3)
# Overlay theoretical exponential
sigma_t = uo2.xs_total(e_thermal)
d_plot = np.linspace(0, max(distances), 200)
ax1.plot(d_plot, sigma_t * np.exp(-sigma_t * d_plot), "r-", lw=2,
         label=rf"$\Sigma_t e^{{-\Sigma_t d}}$, $\Sigma_t$={sigma_t:.2f} cm$^{{-1}}$")
ax1.set_xlabel("Distance to collision (cm)")
ax1.set_ylabel("Probability density")
ax1.set_title("Collision distance sampling (thermal)")
ax1.legend()

# Nuclide hit frequency
from collections import Counter
counts = Counter(nuclides)
zaids = list(counts.keys())
freqs = [counts[z] / n_samples for z in zaids]
ax2.bar(zaids, freqs, edgecolor="black", lw=0.3)
ax2.set_ylabel("Fraction of collisions")
ax2.set_title("Target nuclide sampling (thermal)")
for i, (z, f) in enumerate(zip(zaids, freqs)):
    ax2.text(i, f + 0.01, f"{f:.1%}", ha="center", fontsize=9)

fig.tight_layout()
plt.show()

## Reaction list and per-reaction cross sections

Inspect individual reaction channels for U-235 and plot a few important ones.

In [ ]:
from aleathor.nucdata import reaction_classify

rxns = u235.reactions()
print(f"U-235 has {len(rxns)} reactions:\n")
print(f"  {'MT':>4s}  {'Q (MeV)':>10s}  {'Type':>12s}")
for r in rxns:
    cls = reaction_classify(r["mt"])
    print(f"  {r['mt']:4d}  {r['q_value']:+10.4f}  {cls:>12s}")

In [ ]:
# Plot key reaction cross sections for U-235
mt_labels = {18: "Fission (MT=18)", 102: "(n,γ) (MT=102)", 2: "Elastic (MT=2)"}

fig, ax = plt.subplots(figsize=(10, 6))
for mt, label in mt_labels.items():
    xs = [u235.xs_reaction(mt, e) for e in energies]
    ax.loglog(energies, xs, label=label, lw=0.6)

ax.set_xlabel("Energy (MeV)")
ax.set_ylabel("Cross section (barn)")
ax.set_title("U-235 reaction cross sections")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

## ZAID parsing utility

Decode ZAID strings into their components.

In [ ]:
for zaid in ["92235.80c", "94239.70c", "1001.80c", "8016.80c", "26056.80c"]:
    info = parse_zaid(zaid)
    print(f"  {zaid:>12s}  ->  Z={info['Z']:3d}  A={info['A']:3d}  type={info['type']}")